# 05 · Vector Search：用向量找到片段

现在把前两步接起来：加载知识库并切成片段，再把片段和问题都转成向量，按相似度取前几名。先用内存数组，不引入数据库。

In [1]:
import os
import re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def embed(texts):
    if not client:
        raise RuntimeError("请先配置 LLM_API_KEY 或 OPENAI_API_KEY。")
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)

In [2]:
chunks = load_chunks()
print("知识库片段数：", len(chunks))

if client:
    chunk_vectors = embed([item["text"] for item in chunks])
    chunk_vectors /= np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
    print("向量维度：", chunk_vectors.shape[1])

知识库片段数： 65


## 写一个最小搜索函数

In [3]:
def search(question, top_k=4):
    question_vector = embed([question])[0]
    question_vector /= np.linalg.norm(question_vector)
    scores = chunk_vectors @ question_vector
    indexes = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[index], "score": float(scores[index])} for index in indexes]

if client:
    for result in search("SKU-YG301 瑜伽裤的面料成分是什么？"):
        print(f"{result['score']:.3f}  {result['source']}")
        print(result["text"][:160].replace("\n", " ") + "...")
else:
    print("未搜索：请配置 Embedding API 后运行")

未搜索：请配置 Embedding API 后运行
